# Mybrew CMO Campaign Simulator
## Interactive Budget Optimization & ROI Simulation
This notebook provides an interactive simulation for the Mybrew CMO to:
- Visualize campaign performance across publishers
- Simulate budget reallocation scenarios
- View real-time personalization trigger effectiveness
- Monitor ROI by campaign/publisher/region

In [ ]:
# Cell 1: Load campaign data and create visualizations
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load data from lakehouse
df_campaigns = spark.sql('SELECT * FROM dim_campaign').toPandas()
df_transactions = spark.sql('SELECT * FROM fact_transaction').toPandas()
df_impressions = spark.sql('SELECT * FROM fact_impression').toPandas()
df_customers = spark.sql('SELECT * FROM dim_customer').toPandas()
df_weather = spark.sql('SELECT * FROM dim_weather').toPandas()

print(f'Campaigns: {len(df_campaigns)}')
print(f'Transactions: {len(df_transactions)}')
print(f'Impressions: {len(df_impressions)}')
print(f'Customers: {len(df_customers)}')
print(f'Weather records: {len(df_weather)}')

In [ ]:
# Cell 2: Publisher Performance Comparison
publisher_stats = df_impressions.groupby('publisher').agg(
    impressions=('impression_id', 'count'),
    clicks=('clicked', 'sum'),
    conversions=('converted', 'sum'),
    spend=('spend_amount_gbp', 'sum')
).reset_index()

publisher_stats['CTR'] = publisher_stats['clicks'] / publisher_stats['impressions']
publisher_stats['CPA'] = publisher_stats['spend'] / publisher_stats['conversions'].replace(0, 1)

fig = make_subplots(rows=1, cols=2, subplot_titles=('CTR by Publisher', 'CPA by Publisher'))
fig.add_trace(go.Bar(x=publisher_stats['publisher'], y=publisher_stats['CTR'], name='CTR', marker_color='#0078D4'), row=1, col=1)
fig.add_trace(go.Bar(x=publisher_stats['publisher'], y=publisher_stats['CPA'], name='CPA (GBP)', marker_color='#E74856'), row=1, col=2)
fig.update_layout(title='Publisher Performance - Winter Warmth Campaign', height=400, showlegend=False)
fig.show()

In [ ]:
# Cell 3: Budget Reallocation Simulator
print('=== BUDGET REALLOCATION SIMULATOR ===')
print()

# Current allocation
current_budget = {'Meta': 2400000, 'Google': 1800000, 'Amazon': 600000}
current_roas = {'Meta': 4.13, 'Google': 2.76, 'Amazon': 3.40}

print('CURRENT ALLOCATION:')
for pub, budget in current_budget.items():
    print(f'  {pub}: GBP{budget:,.0f} (ROAS: {current_roas[pub]}x)')

total_budget = sum(current_budget.values())
print(f'  Total: GBP{total_budget:,.0f}')
print()

# Simulation: Move 120K from Google to Meta
realloc_amount = 120000
new_budget = current_budget.copy()
new_budget['Google'] -= realloc_amount
new_budget['Meta'] += realloc_amount

# Project new conversions
google_cpa = 32.00
meta_cpa = 21.14
lost_conversions = realloc_amount / google_cpa
gained_conversions = realloc_amount / meta_cpa
net_conversions = gained_conversions - lost_conversions

print('RECOMMENDED REALLOCATION:')
print(f'  Move GBP{realloc_amount:,.0f} from Google Display -> Meta Stories')
print()
print('PROJECTED IMPACT:')
for pub, budget in new_budget.items():
    print(f'  {pub}: GBP{budget:,.0f}')
print(f'  Net new conversions: +{net_conversions:,.0f}')
print(f'  Blended CPA improvement: -GBP2.10')
print(f'  Projected additional revenue: GBP{net_conversions * 4.75:,.0f}')
print()
print('STATUS: Ready to execute')

In [ ]:
# Cell 4: Personalization Trigger Simulation
print('=== PERSONALIZATION TRIGGERS (London, Today) ===')
print()

# Weather check
london_weather = df_weather[df_weather['city'] == 'London'].sort_values('date', ascending=False).iloc[0]
print(f'Current Weather: {london_weather["condition"]}, {london_weather["temp_c"]}C')
print()

# Trigger 1: Cold weather
if london_weather['temp_c'] < 5:
    cold_eligible = df_customers[(df_customers['city'] == 'London') & (df_customers['opted_in'] == True)]
    print(f'TRIGGER: Cold Weather Offer (temp < 5C)')
    print(f'  Eligible customers: {len(cold_eligible)}')
    print(f'  Offer: Warm Hazelnut Flat White, 20% off')
    print(f'  Est. redemption rate: 14%')
    print(f'  Est. revenue: GBP{len(cold_eligible) * 0.14 * 4.75:,.0f}')
    print()

# Trigger 2: Win-back
import datetime

# Use today's date dynamically instead of hard-coding
# and handle both datetime and date values safely

# If you want a fixed demo date, replace this with:
# today = datetime.date(2026, 7, 7)

today = datetime.date.today()

lapsed = df_customers[(df_customers['opted_in'] == True)].copy()

def compute_days_since_last_visit(x):
    """Compute days since last visit from a pandas Timestamp, datetime or date.
    Returns a large number (999) when value is missing."""
    if pd.isna(x):
        return 999
    # If it's a pandas Timestamp or datetime, use .date(); if it's already a date, use as-is
    if hasattr(x, 'date'):
        last_visit_date = x.date()
    else:
        last_visit_date = x
    return (today - last_visit_date).days

lapsed['days_since'] = lapsed['last_visit'].apply(compute_days_since_last_visit)

lapsed_30d = lapsed[lapsed['days_since'] >= 30]
print(f'TRIGGER: Win-Back (30+ days inactive)')
print(f'  Eligible customers: {len(lapsed_30d)}')
print(f'  Offer: Free drink voucher')
print(f'  Est. return rate: 21%')
print(f'  Est. revenue: GBP{len(lapsed_30d) * 0.21 * 5.25:,.0f}')
print()

# Trigger 3: Tier nudge
tier_nudge = df_customers[(df_customers['tier'] == 'Silver') & (df_customers['tier_points'] >= 450)]
print(f'TRIGGER: Tier Nudge (Silver near Gold, 450+ pts)')
print(f'  Eligible customers: {len(tier_nudge)}')
print(f'  Message: You are X points from Gold!')
print(f'  Est. conversion to Gold: 30%')

In [ ]:
# Cell 5: ROI Dashboard Visualization
# Simulate ROI data
roi_data = pd.DataFrame({
    'Campaign': ['Winter Warmth', 'Winter Warmth', 'Winter Warmth', 'Summer Sips', 'Win-Back Q3', 'Tier Nudge'],
    'Publisher': ['Meta', 'Google', 'Amazon', 'Meta', 'Direct', 'Direct'],
    'Spend_GBP': [890000, 720000, 210000, 1200000, 180000, 45000],
    'Revenue_GBP': [3740000, 2016000, 714000, 4100000, 890000, 525000],
    'Conversions': [42100, 22500, 7800, 38000, 12400, 2100]
})

roi_data['ROAS'] = roi_data['Revenue_GBP'] / roi_data['Spend_GBP']
roi_data['CPA'] = roi_data['Spend_GBP'] / roi_data['Conversions']
roi_data['ROI_pct'] = ((roi_data['Revenue_GBP'] - roi_data['Spend_GBP']) / roi_data['Spend_GBP'] * 100).round(1)

# ROAS comparison chart
fig = px.bar(roi_data, x='Campaign', y='ROAS', color='Publisher',
             title='ROAS by Campaign & Publisher',
             color_discrete_map={'Meta': '#0078D4', 'Google': '#34A853', 'Amazon': '#FF9900', 'Direct': '#6B4C9A', 'NYT': '#333333'},
             barmode='group')
fig.add_hline(y=1, line_dash='dash', line_color='red', annotation_text='Break-even')
fig.update_layout(height=450)
fig.show()

print()
print('=== LIVE ROI SUMMARY ===')
total_rev = roi_data['Revenue_GBP'].sum()
total_spend = roi_data['Spend_GBP'].sum()
print(f'Total Attributed Revenue: GBP{total_rev:,.0f}')
print(f'Total Ad Spend: GBP{total_spend:,.0f}')
print(f'Overall ROAS: {total_rev/total_spend:.2f}x')
print(f'Overall ROI: {(total_rev-total_spend)/total_spend*100:.1f}%')